In [ ]:

!pip -q install transformers torch sentencepiece librosa soundfile SpeechRecognition

import os
import re
import numpy as np
import librosa
import soundfile as sf
import speech_recognition as sr

from transformers import pipeline

print("=" * 75)
print("             🎤 VOICE-BASED STRESS DETECTION")
print("                  AI SPEECH ANALYZER")
print("=" * 75)



print("\n⏳ Loading AI models...")

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)

print("✅ AI models loaded successfully!")



print("\n" + "-" * 75)
print("📁 UPLOAD YOUR VOICE RECORDING")
print("-" * 75)

print("""
Supported formats:
• WAV
• MP3
• M4A
• FLAC
""")

from google.colab import files

uploaded = files.upload()

if len(uploaded) == 0:
    print("❌ No audio file uploaded.")
    raise SystemExit

audio_file = list(uploaded.keys())[0]

print("\n✅ Uploaded:", audio_file)



print("\n⏳ Processing audio...")

try:

    audio, sample_rate = librosa.load(
        audio_file,
        sr=16000,
        mono=True
    )

except Exception as e:

    print("❌ Could not process audio.")
    print("Error:", e)
    raise SystemExit



duration = len(audio) / sample_rate



try:

    trimmed_audio, index = librosa.effects.trim(
        audio,
        top_db=30
    )

except:

    trimmed_audio = audio



rms = librosa.feature.rms(
    y=trimmed_audio
)[0]

average_energy = float(
    np.mean(rms)
)

energy_std = float(
    np.std(rms)
)



try:

    f0, voiced_flag, voiced_prob = librosa.pyin(
        trimmed_audio,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sample_rate
    )

    valid_pitch = f0[~np.isnan(f0)]

    if len(valid_pitch) > 0:

        average_pitch = float(
            np.mean(valid_pitch)
        )

        pitch_std = float(
            np.std(valid_pitch)
        )

    else:

        average_pitch = 0
        pitch_std = 0

except:

    average_pitch = 0
    pitch_std = 0



zcr = librosa.feature.zero_crossing_rate(
    trimmed_audio
)[0]

average_zcr = float(
    np.mean(zcr)
)




print("\n🎤 Converting speech to text...")

recognizer = sr.Recognizer()

wav_file = "processed_interview.wav"

sf.write(
    wav_file,
    trimmed_audio,
    sample_rate
)

try:

    with sr.AudioFile(wav_file) as source:

        recorded_audio = recognizer.record(
            source
        )

    transcript = recognizer.recognize_google(
        recorded_audio
    )

    print("✅ Speech converted successfully!")

except Exception as e:

    print("⚠️ Speech recognition failed.")
    print("Using empty transcript.")

    transcript = ""



if transcript.strip():

    clean_text = re.sub(
        r"\s+",
        " ",
        transcript
    ).strip()

    lower_text = clean_text.lower()

else:

    clean_text = "No speech transcript available."

    lower_text = ""



if transcript.strip():

    sentiment_result = sentiment_model(
        clean_text[:512]
    )[0]

    sentiment_label = sentiment_result["label"]

    sentiment_confidence = (
        sentiment_result["score"] * 100
    )

    if sentiment_label == "POSITIVE":

        sentiment = "Positive 😊"

    else:

        sentiment = "Negative 😟"

else:

    sentiment = "Unavailable"

    sentiment_confidence = 0



if transcript.strip():

    emotion_result = emotion_model(
        clean_text[:512]
    )[0][0]

    emotion_label = emotion_result["label"]

    emotion_confidence = (
        emotion_result["score"] * 100
    )

    emotion_names = {

        "anger": "Angry 😠",
        "disgust": "Disgusted 🤢",
        "fear": "Fearful 😨",
        "joy": "Happy 😊",
        "neutral": "Neutral 😐",
        "sadness": "Sad 😢",
        "surprise": "Surprised 😮"

    }

    emotion = emotion_names.get(
        emotion_label.lower(),
        emotion_label.capitalize()
    )

else:

    emotion = "Unavailable"

    emotion_confidence = 0



if transcript.strip() and duration > 0:

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        lower_text
    )

    word_count = len(words)

    speech_rate = (
        word_count / (duration / 60)
    )

else:

    word_count = 0
    speech_rate = 0



try:

    intervals = librosa.effects.split(
        audio,
        top_db=30
    )

    speech_time = sum(
        (end - start) / sample_rate
        for start, end in intervals
    )

    pause_time = max(
        0,
        duration - speech_time
    )

    pause_percentage = (
        pause_time / duration
    ) * 100 if duration > 0 else 0

except:

    pause_time = 0
    pause_percentage = 0



stress_score = 0

stress_indicators = []



if speech_rate > 180:

    stress_score += 20

    stress_indicators.append(
        "Fast speaking rate"
    )

elif speech_rate > 160:

    stress_score += 10

    stress_indicators.append(
        "Slightly fast speaking rate"
    )

elif speech_rate < 80 and speech_rate > 0:

    stress_score += 10

    stress_indicators.append(
        "Very slow speaking rate"
    )



if average_pitch > 0:

    pitch_variation = (
        pitch_std / average_pitch
    ) * 100

    if pitch_variation > 35:

        stress_score += 15

        stress_indicators.append(
            "High pitch variation"
        )

    elif pitch_variation > 25:

        stress_score += 8

        stress_indicators.append(
            "Moderate pitch variation"
        )

else:

    pitch_variation = 0



if average_energy > 0:

    energy_variation = (
        energy_std / average_energy
    ) * 100

    if energy_variation > 60:

        stress_score += 15

        stress_indicators.append(
            "High voice-energy variation"
        )

    elif energy_variation > 40:

        stress_score += 8

        stress_indicators.append(
            "Moderate voice-energy variation"
        )

else:

    energy_variation = 0



if pause_percentage > 40:

    stress_score += 15

    stress_indicators.append(
        "Long or frequent pauses"
    )

elif pause_percentage > 25:

    stress_score += 8

    stress_indicators.append(
        "Moderate pauses"
    )



if sentiment == "Negative 😟":

    stress_score += 10

    stress_indicators.append(
        "Negative sentiment"
    )




if emotion_label if transcript.strip() else "":

    if emotion_label.lower() in [
        "fear",
        "anger",
        "sadness"
    ]:

        stress_score += 15

        stress_indicators.append(
            "Stress-related emotional indicators"
        )


# Limit score
stress_score = min(
    100,
    stress_score
)



if stress_score >= 70:

    stress_level = "HIGH STRESS RISK 🔴"

elif stress_score >= 40:

    stress_level = "MODERATE STRESS RISK 🟠"

else:

    stress_level = "LOW STRESS RISK 🟢"



confidence_score = 70

if speech_rate >= 100 and speech_rate <= 160:

    confidence_score += 10

if pause_percentage < 25:

    confidence_score += 10

if sentiment == "Positive 😊":

    confidence_score += 5

if emotion_label if transcript.strip() else "":

    if emotion_label.lower() == "fear":

        confidence_score -= 10

    if emotion_label.lower() == "sadness":

        confidence_score -= 5

confidence_score = max(
    0,
    min(100, confidence_score)
)



recommendations = []

if speech_rate > 160:

    recommendations.append(
        "Try speaking more slowly and clearly."
    )

if speech_rate < 80 and speech_rate > 0:

    recommendations.append(
        "Try maintaining a slightly faster and natural speaking pace."
    )

if pause_percentage > 30:

    recommendations.append(
        "Practice reducing long pauses between sentences."
    )

if pitch_variation > 35:

    recommendations.append(
        "Practice calm breathing to maintain a steadier voice."
    )

if energy_variation > 50:

    recommendations.append(
        "Maintain consistent voice volume while speaking."
    )

if sentiment == "Negative 😟":

    recommendations.append(
        "Use positive and confident language."
    )

if not recommendations:

    recommendations.append(
        "Voice characteristics appear relatively stable. Continue practicing."
    )



print("\n")
print("=" * 75)
print("                    📊 STRESS ANALYSIS REPORT")
print("=" * 75)



print("\n📝 SPEECH TRANSCRIPT")
print("-" * 75)

print(clean_text)



print("\n🧠 STRESS DETECTION")
print("-" * 75)

print(
    "Stress Risk Score:",
    f"{stress_score}/100"
)

print(
    "Stress Level     :",
    stress_level
)



print("\n😊 SENTIMENT ANALYSIS")
print("-" * 75)

print(
    "Sentiment:",
    sentiment
)

print(
    "Confidence:",
    f"{sentiment_confidence:.2f}%"
)



print("\n🎭 EMOTION ANALYSIS")
print("-" * 75)

print(
    "Emotion:",
    emotion
)

print(
    "Confidence:",
    f"{emotion_confidence:.2f}%"
)



print("\n🔊 VOICE FEATURES")
print("-" * 75)

print(
    "Audio Duration:",
    f"{duration:.2f} seconds"
)

print(
    "Speaking Rate:",
    f"{speech_rate:.2f} words/minute"
)

print(
    "Average Voice Energy:",
    f"{average_energy:.4f}"
)

print(
    "Energy Variation:",
    f"{energy_variation:.2f}%"
)

print(
    "Average Pitch:",
    f"{average_pitch:.2f} Hz"
)

print(
    "Pitch Variation:",
    f"{pitch_variation:.2f}%"
)

print(
    "Pause Time:",
    f"{pause_time:.2f} seconds"
)

print(
    "Pause Percentage:",
    f"{pause_percentage:.2f}%"
)



print("\n💪 CONFIDENCE INDICATOR")
print("-" * 75)

print(
    "Confidence Score:",
    f"{confidence_score}/100"
)



print("\n🚨 DETECTED STRESS INDICATORS")
print("-" * 75)

if stress_indicators:

    for indicator in stress_indicators:

        print("⚠️", indicator)

else:

    print(
        "✅ No major stress indicators detected."
    )



print("\n💡 RECOMMENDATIONS")
print("-" * 75)

for recommendation in recommendations:

    print("•", recommendation)



print("\n")
print("=" * 75)

if stress_score >= 70:

    print(
        "⚠️ RESULT: HIGH STRESS INDICATORS DETECTED"
    )

elif stress_score >= 40:

    print(
        "⚠️ RESULT: MODERATE STRESS INDICATORS DETECTED"
    )

else:

    print(
        "✅ RESULT: LOW STRESS INDICATORS DETECTED"
    )

print("=" * 75)


print("""
⚠️ IMPORTANT:

This is an educational AI project.

The system estimates stress-related indicators from voice
features such as speaking rate, pitch, energy, pauses and
text sentiment.

It cannot diagnose stress, anxiety, depression, or any medical
condition. Voice characteristics can vary naturally because of
accent, language, microphone quality, background noise,
personality and speaking style.

For a stronger academic project, a labeled speech dataset
should be used to train and evaluate a dedicated stress-
classification machine-learning model.
""")

print("\n🎤 ANALYSIS COMPLETED SUCCESSFULLY!")